Title: Combine_future.ipynb

Purpose: Combine all datasets for the future

Author: Onno Nennecke on 09.06.2025 Modified: 03.11.2026

Input data: 

- Model output data
    - This file lies here: '/climca/people/onennecke/model_output/not_bias_corrected/future_full_year_one_time_2005_2024_ref/'  # future_full_year

Output data:

- Combined model output: model_output_all.nc
    - This file lies here: /climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future_2005_2024_ref.nc  or model_output_all_future.nc

In [1]:
# Importing libraries
import xarray as xr
import numpy as np
import os
import glob

In [2]:
#### path = '/climca/people/onennecke/model_output/bias_corrected_masked_ibicus/full_year/'
# path = '/climca/people/onennecke/model_output/not_bias_corrected/full_year/'
# path = '/climca/people/onennecke/model_output/not_bias_corrected/future_full_year_one_time/'
path = '/climca/people/onennecke/model_output/not_bias_corrected/future_full_year_one_time_2005_2024_ref/'


files = sorted(glob.glob(os.path.join(path, '*.nc')))
ts_datasets = xr.open_mfdataset(files, combine='nested', concat_dim='ESM_run')

ts_datasets.load()

<xarray.Dataset> Size: 59MB
Dimensions:        (ESM_run: 192, time: 3650)
Coordinates:
    crs            int64 8B 4326
    gridtype       <U6 24B 'lonlat'
    ESM            (ESM_run) <U13 10kB 'ACCESS-CM2' ... 'UKESM1-0-LL'
    run            (ESM_run) <U12 9kB 'r1i1p1f1' 'r4i1p1f1' ... 'r8i1p1f2'
  * ESM_run        (ESM_run) <U23 18kB 'ACCESS-CM2_r1i1p1f1' ... 'UKESM1-0-LL...
    country        float64 8B 9.0
    period         <U4 16B 'week'
  * time           (time) datetime64[ns] 29kB 2041-01-01 ... 2050-12-31
Data variables:
    temp           (ESM_run, time) float64 6MB 1.053 3.103 3.548 ... 2.553 2.669
    demand         (ESM_run, time) float64 6MB 1.507e+03 1.482e+03 ... 1.487e+03
    sfcWind        (ESM_run, time) float64 6MB 6.256 7.448 7.004 ... 3.361 4.569
    rsds           (ESM_run, time) float32 3MB 14.55 18.67 21.11 ... 26.1 24.82
    tas            (ESM_run, time) float32 3MB 1.577 3.426 3.581 ... 2.589 2.408
    tasmax         (ESM_run, time) float32 3MB 2.851 4.208 4.731 ... 4.264 3.786
    wind_off_prod  (ESM_run, time) float64 6MB 135.5 86.95 89.92 ... 24.32 70.92
    wind_on_prod   (ESM_run, time) float64 6MB 430.5 677.6 445.9 ... 49.53 140.7
    solar_prod     (ESM_run, time) float64 6MB 24.34 28.8 32.36 ... 46.84 46.45
    total_prod     (ESM_run, time) float64 6MB 590.3 793.4 568.2 ... 120.7 258.1
    Netto          (ESM_run, time) float64 6MB -916.6 -688.1 ... -1.229e+03
    Residual_load  (ESM_run, time) float64 6MB 916.6 688.1 ... 1.229e+03

In [3]:
ts_datasets.coords['doy'] = (('time',), np.tile(np.arange(1, 366), 10))
ts_datasets

combined_ds = ts_datasets.copy()

### Save the data

In [4]:
encoding = {}

all_vars = {**combined_ds.data_vars, **combined_ds.coords}

for var, da in all_vars.items():
    if da.dtype.kind in {"U", "S"}:
        values = da.values.astype(str)
        if values.ndim == 0:
            maxlen = len(str(values))
        else:
            maxlen = max(map(len, values.flatten()))
        encoding[var] = {"dtype": f"U{maxlen}"}
encoding

{'gridtype': {'dtype': 'U6'},
 'ESM': {'dtype': 'U13'},
 'run': {'dtype': 'U12'},
 'ESM_run': {'dtype': 'U23'},
 'period': {'dtype': 'U4'}}

In [5]:
# Save dataset
# combined_ds.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future.nc', encoding=encoding)
combined_ds.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future_2005_2024_ref.nc', encoding=encoding)
# combined_ds.to_netcdf('/climca/people/onennecke/model_output/not_bias_corrected/model_output_all_future.nc')